#### Load base

In [ ]:
pip install neuralforecast==1.7.6

In [ ]:
import datetime
import pandas as pd
import numpy as np
import re
import time, datetime
from google.colab import drive
from tqdm import tqdm_notebook
import warnings
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
def smape_np(A, F):
    return 100/len(A) * np.sum(2 * np.abs(F - A) / (np.abs(A) + np.abs(F)))

In [ ]:
from datetime import timedelta
def subtract_days_from_date(date, days):
    """Subtract days from a date and return the date.

    Args:
        date (string): Date string in YYYY-MM-DD format.
        days (int): Number of days to subtract from date

    Returns:
        date (date): Date in YYYY-MM-DD with X days subtracted.
    """

    subtracted_date = pd.to_datetime(date) - timedelta(days=days)
    subtracted_date = subtracted_date.strftime("%Y-%m-%d")

    return subtracted_date

In [ ]:
import numpy as np

def wql(y_true, y_pred, quantile, weights=None):
    """
    Compute Weighted Quantile Loss (WQL) for a given quantile.

    Parameters:
        y_true (array-like): Actual values (n_samples,).
        y_pred (array-like): Predicted quantile values (n_samples,).
        quantile (float): Quantile to evaluate (e.g., 0.1 or 0.9).
        weights (array-like or None): Weights for each observation (n_samples,). If None, uniform weights are used.

    Returns:
        float: Weighted Quantile Loss.
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    if weights is None:
        weights = np.ones_like(y_true)  # Uniform weights

    # Quantile Loss components
    errors = y_true - y_pred
    quantile_loss = np.maximum(quantile * errors, (quantile - 1) * errors)

    # Weighted Quantile Loss
    wql = np.sum(weights * quantile_loss) / np.sum(weights)
    return wql


In [ ]:
from scipy.stats import spearmanr

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# plt.style.use('fivethirtyeight')

In [ ]:
import pandas as pd
# prophet_df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_ts_data_v1.csv", index=False)
prophet_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_ts_data_v1.csv")
final_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_ts_data_v2.csv")
# df_plot = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/prophet_toplot.csv")
# emeu = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/PersonalFinance/extra_market_data.csv')
# emeu['Timestamp'] = emeu.DATE.apply(lambda x: subtract_days_from_date(x, 1))
# emeu['Timestamp'] = pd.to_datetime(emeu['Timestamp'])
# emeu

In [ ]:
prophet_df['unique_id'] = 1
prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])
final_df['unique_id'] = 1
final_df['ds'] = pd.to_datetime(final_df['ds'])

#### Multi step regression LSTM 7-day

In [ ]:
result_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/result_df_dl_new_2025.csv")
result_df['ds'] = pd.to_datetime(result_df['ds'])
result_df.head(3)

In [ ]:
result_df[list(result_df.columns[0:20])]

In [ ]:
final_metric = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/final_metric_dl_new_2025.csv")
final_metric['RUN_DAY'] = pd.to_datetime(final_metric['RUN_DAY'])

In [ ]:
final_metric.mean(numeric_only=True)

In [ ]:
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM
from neuralforecast.losses.pytorch import MQLoss

from datetime import date, timedelta, datetime
import time

from tqdm import tqdm
from collections import Counter

In [ ]:
FCST_START = date(2022, 2, 21)
FCST_END = date(2022, 5, 26)
DAY_LIST = pd.date_range(start = FCST_START, end = FCST_END, freq = 'D')

In [ ]:
hyperparams = {
    'input_size': 90,
    'h': 30,
    'learning_rate': 1e-4,
    'max_steps': 20000,
    'random_seed': 1,
    'loss': MQLoss(level=[80]),
    'early_stop_patience_steps': 5,
    'val_check_steps': 200,
    'logger': False,
    'enable_progress_bar': True,
    'enable_model_summary': False
                       }


In [ ]:
# base model
smape_base = []
spear_base = []
wql_base = []
feature_importance_base = []
Y_hat_df = pd.DataFrame({})
metric_df = pd.DataFrame({})

tik = time.time()
for CUTOFF_DAY in tqdm(DAY_LIST):

    ############################
    ######### Modeling #########
    ############################ early_stop_patience_steps = 5,
    Y_train_df = prophet_df[prophet_df.ds<CUTOFF_DAY].copy()


    # Fit and predict with LSTM
    models = [
            LSTM(**hyperparams)
            ]

    nf = NeuralForecast(models=models, freq='D', local_scaler_type='robust')

    nf.fit(df=Y_train_df, val_size=30)

    Y_hat_df_temp = nf.predict(df=Y_train_df).reset_index()

    #############################
    ######### Data Prep #########
    #############################
    Y_hat_df_temp = Y_hat_df_temp.merge(prophet_df, on=['ds', 'unique_id'], how='left')
    Y_hat_df_temp['y'].fillna(np.nan, inplace=True)
    Y_hat_df_temp['RUN_DAY'] = Y_hat_df_temp['ds'].min().strftime('%Y-%m-%d')

    ###################################
    ######### Forecasts Store #########
    ###################################
    Y_hat_df = pd.concat([Y_hat_df, Y_hat_df_temp])

    ##########################################
    ######### Get Feature Importance #########
    ##########################################
    current_dict = dict(nf.models[0].feature_importances()['Past variable importance over time'].mean())
    feature_importance_base.append(current_dict)

    # calculate wql
    quantile_loss_50 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-median'], 0.5)
    quantile_loss_90 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-hi-80'], 0.9)
    quantile_loss_10 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-lo-80'], 0.1)
    avg_wql = (quantile_loss_50 + quantile_loss_90 + quantile_loss_10)/3

    # put metrics together
    metric_df_temp = pd.DataFrame({'LSTM_QUANTIL_BASE_SMAPE': [smape_np(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-median'])],
                                   'LSTM_QUANTIL_BASE_SC': [spearmanr(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-median'])[0]],
                                   'LSTM_QUANTIL_BASE_WQL': avg_wql,
                                   'RUN_DAY': [Y_hat_df_temp['ds'].min().strftime('%Y-%m-%d')]})

    metric_df = pd.concat([metric_df, metric_df_temp])




tok = time.time()
EXECUTION_TIME = time.strftime('%H:%M:%Ss', time.gmtime(tok-tik))
print(EXECUTION_TIME)


In [ ]:
metric_df.mean(numeric_only=True)

In [ ]:
metric_df['RUN_DAY'] = pd.to_datetime(metric_df['RUN_DAY'])
final_metric['RUN_DAY'] = pd.to_datetime(final_metric['RUN_DAY'])

In [ ]:
final_metric = final_metric.merge(metric_df, on=['RUN_DAY'], how='left')
final_metric.to_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/metric_df_dl_new_2025.csv", index=False)

In [ ]:
result_df = result_df.merge(Y_hat_df, on=['unique_id', 'ds', 'RUN_DAY', 'y'], how='left')
result_df.to_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/result_df_dl_new_2025.csv", index=False)

In [ ]:
hyperparams = {
    'input_size': 90,
    'h': 30,
    'learning_rate': 1e-4,
    'max_steps': 20000,
    'random_seed': 1,
    'loss': MQLoss(level=[80]),
    'early_stop_patience_steps': 5,
    'val_check_steps': 100,
    'hist_exog_list': ['0', '158', '233', '272', '30'],
    'logger': False,
    'enable_progress_bar': True,
    'enable_model_summary': False
                       }


In [ ]:
# Base model

smape_exo = []
spear_exo = []
wql_base = []
Y_hat_df_exo = pd.DataFrame({})
feature_importance_exo = []
metric_df_exo = pd.DataFrame({})

tik = time.time()

for CUTOFF_DAY in tqdm(DAY_LIST):
    ############################
    ######### Modeling #########
    ############################
    Y_train_df = final_df[final_df.ds < CUTOFF_DAY].copy()

    # Fit and predict with LSTM
    models = [
        LSTM(**hyperparams)
    ]

    nf = NeuralForecast(models=models, freq='D', local_scaler_type='robust')
    nf.fit(df=Y_train_df, val_size=30)

    Y_hat_df_temp = nf.predict(df=Y_train_df).reset_index()

    #############################
    ######### Data Prep #########
    #############################
    Y_hat_df_temp = Y_hat_df_temp.merge(final_df, on=['ds', 'unique_id'], how='left')
    Y_hat_df_temp['y'].fillna(np.nan, inplace=True)
    Y_hat_df_temp['RUN_DAY'] = Y_hat_df_temp['ds'].min().strftime('%Y-%m-%d')

    ###################################
    ######### Forecasts Store #########
    ###################################
    Y_hat_df_exo = pd.concat([Y_hat_df_exo, Y_hat_df_temp])

    ##########################################
    ######### Get Feature Importance #########
    ##########################################
    # Calculate wql
    quantile_loss_50 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-median'], 0.5)
    quantile_loss_90 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-hi-80'], 0.9)
    quantile_loss_10 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-lo-80'], 0.1)
    avg_wql = (quantile_loss_50 + quantile_loss_90 + quantile_loss_10) / 3

    # Put metrics together
    metric_df_temp = pd.DataFrame({
        'LSTM_QUANTIL_BASE_SMAPE_EXO': [smape_np(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-median'])],
        'LSTM_QUANTIL_BASE_SC_EXO': [spearmanr(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-median'])[0]],
        'LSTM_QUANTIL_BASE_WQL_EXO': avg_wql,
        'RUN_DAY': [Y_hat_df_temp['ds'].min().strftime('%Y-%m-%d')]
    })

    metric_df_exo = pd.concat([metric_df_exo, metric_df_temp])

tok = time.time()
EXECUTION_TIME = time.strftime('%H:%M:%Ss', time.gmtime(tok - tik))
print(EXECUTION_TIME)

In [ ]:
metric_df_exo.mean(numeric_only=True)

In [ ]:
metric_df_exo['RUN_DAY'] = pd.to_datetime(metric_df_exo['RUN_DAY'])
final_metric = final_metric.merge(metric_df_exo, on=['RUN_DAY'], how='left')
final_metric.to_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/final_metric_dl_new_2025.csv", index=False)

In [ ]:
result_df = result_df.merge(Y_hat_df_exo.rename(columns={'LSTM-median': 'LSTM-MEDIAN-EXO', 'LSTM-lo-80': 'LSTM-LO-80-EXO', 'LSTM-hi-80': 'LSTM-HI-80-EXO'}), on=['ds', 'unique_id', 'y', 'RUN_DAY'], how='left')
result_df.to_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/result_df_dl_new_2025.csv", index=False)

In [ ]:
plot_df = result_df
plot_df['RUN_DAY'] = plot_df['RUN_DAY'].astype(str)
plot_df.sort_values(['RUN_DAY', 'ds', 'unique_id'],  inplace=True)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(20,5), dpi=300)
plt.plot(plot_df['RUN_DAY'].unique(), plot_df.groupby('RUN_DAY')['y'].mean(), c='black', label='True')
plt.plot(plot_df['RUN_DAY'].unique(), plot_df.groupby('RUN_DAY')['LSTM-median'].mean(), c='blue', label='LSTM Forecast')
plt.plot(plot_df['RUN_DAY'].unique(), plot_df.groupby('RUN_DAY')['LSTM-MEDIAN-EXO'].mean(), c='orange', label='LSTM_EXO Forecast')
plt.fill_between(x=plot_df['RUN_DAY'].unique(),
                 y1=plot_df.groupby('RUN_DAY')['LSTM-lo-80'].mean(),
                 y2=plot_df.groupby('RUN_DAY')['LSTM-hi-80'].mean(),
                 alpha=0.4, label='level 90')
plt.fill_between(x=plot_df['RUN_DAY'].unique(),
                 y1=plot_df.groupby('RUN_DAY')['LSTM-LO-80-EXO'].mean(),
                 y2=plot_df.groupby('RUN_DAY')['LSTM-HI-80-EXO'].mean(),
                 alpha=0.4, label='level 90')
plt.title(f'Average 30-Day Forecasts')
plt.xticks(plot_df['RUN_DAY'].unique()[::7])
plt.legend()
plt.grid()
plt.show()

In [ ]:
final_metric[final_metric.RUN_DAY>='2022-03-07'].mean(numeric_only=True)